In [13]:
from pathlib import Path
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format

from Scripts.weekly_view import build_weekly_view
from Scripts.schedule import build_season_schedule
from Scripts.features import compute_rolling_stats
from Scripts.preseason_runner import run_preseason_reports
from Scripts.preseason import build_preseason_package, init_shortlist_template, load_shortlist, render_preseason_player_report, get_shortlist_path

from Scripts.data_io import load_rounds, load_odds_and_results, load_course_fit, load_player_skills

from Scripts.visuals import (
    subset_weekly_for_players,
    plot_ev_current_vs_future_max,
    plot_course_vs_players_radar_from_skills,
    print_event_history_for_player,
)

from Scripts.notebook_support import (
    configure_notebook_display,
    build_default_paths,
    load_course_importance,
    apply_used_filter_to_weekly,
    build_condensed_table,
    filter_top_tables_by_ids,
    log_pick,
    preview_pick_result,
    add_player_names_to_picks_file,
    print_preseason_reports,
)

configure_notebook_display()

SEASON_YEAR = 2024
# TOP_N_PRESEASON = 15
PROJECT_ROOT = Path("/Users/joshmacbook/python_projects/OAD")

paths = build_default_paths(PROJECT_ROOT, SEASON_YEAR)
course_imp_df = load_course_importance(paths.course_imp_path)

path = get_shortlist_path(SEASON_YEAR)
if os.path.exists(path):
    os.remove(path)

init_shortlist_template(SEASON_YEAR, top_n=TOP_N_PRESEASON)
print("Shortlist template recreated:", path)

Shortlist template recreated: /Users/joshmacbook/python_projects/OAD/Data/in Use/preseason_shortlist_2024.csv


In [14]:
rounds_df = load_rounds()
odds_df = load_odds_and_results()
course_fit_df = load_course_fit(SEASON_YEAR)
player_skills_df = load_player_skills(SEASON_YEAR)
schedule_df = build_season_schedule(SEASON_YEAR)

In [15]:
pre_pkg = build_preseason_package(season=SEASON_YEAR, top_n=TOP_N_PRESEASON)
longlist = pre_pkg["longlist"]
big_hist_df = pre_pkg["major_sig_history"]

preseason_as_of = schedule_df["event_date"].min()

baseline_df = compute_rolling_stats(
    rounds_df=rounds_df,
    as_of_date=preseason_as_of,
    dg_ids=longlist["dg_id"].dropna().astype(int).tolist(),
    windows=(40, 24, 12),
)

shortlist_template_path = init_shortlist_template(season=SEASON_YEAR, top_n=TOP_N_PRESEASON)
shortlist_df = load_shortlist(SEASON_YEAR)

In [16]:
ctx = run_preseason_reports(
    season_year=SEASON_YEAR,
    top_n=15,
    windows=(40, 24, 12),
)

longlist = ctx["longlist"]
big_hist_df = ctx["big_hist_df"]
baseline_df = ctx["baseline_df"]

Preseason long list:


,dg_id,hist_rounds,dec_wgh_sg,is_liv,target_season
0,10091,159,2.49,False,2024
1,18417,162,2.41,False,2024
2,19195,166,2.16,False,2024
3,19895,153,1.93,False,2024
4,15466,144,1.91,False,2024
5,18841,181,1.84,False,2024
6,14796,175,1.59,False,2024
7,12294,180,1.54,False,2024
8,15856,132,1.53,True,2024
9,18079,132,1.47,True,2024


Player dg_id 10091
Star power (dec_wgh_sg): 2.492

Baseline strokes gained (L40/L24/L12, pre season):
  L40: OTT +0.911, APP +0.868, ARG +0.207, PUTT +0.345, TOTAL +2.235 (n=40)  (rank 1/15 in shortlist)
  L24: OTT +0.682, APP +0.128, ARG +0.144, PUTT +0.728, TOTAL +1.930 (n=24)  (rank 2/15 in shortlist)
  L12: OTT +nan, APP +nan, ARG +nan, PUTT +nan, TOTAL +1.776 (n=12)  (rank 4/15 in shortlist)

Big-event history since 2017 (MAJOR + SIGNATURE, no playoffs):
  Masters Tournament (event_id 14)
    Starts: 6, Cuts made: 5, Top10: 4, Top25: 5
    Best finish: 2, Worst: 21, Avg: 8.0, Avg SG event: +7.58
  PGA Championship (event_id 33)
    Starts: 7, Cuts made: 7, Top10: 3, Top25: 4
    Best finish: 7, Worst: 50, Avg: 25.3, Avg SG event: +6.99
  The Open Championship (event_id 100)
    Starts: 6, Cuts made: 5, Top10: 4, Top25: 4
    Best finish: 2, Worst: 46, Avg: 12.2, Avg SG event: +8.47
  U.S. Open (event_id 26)
    Starts: 6, Cuts made: 4, Top10: 4, Top25: 4
    Best finish: 2, Worst:

In [22]:
INDEX = 0
EVENT_ID = int(schedule_df.iloc[INDEX]["event_id"])

weekly_raw = build_weekly_view(SEASON_YEAR, EVENT_ID)  # unfiltered
weekly, used_ids = apply_used_filter_to_weekly(weekly_raw.copy(), paths.picks_path)

print(f"Building weekly view for {SEASON_YEAR}, event_id = {EVENT_ID}")
print(f"Used players from manual picks: {len(used_ids)} dg_ids filtered out.")

display(weekly["schedule_row"])

condensed = build_condensed_table(weekly)
display(condensed.
        head(20))

Building weekly view for 2024, event_id = 6
Used players from manual picks: 1 dg_ids filtered out.


,year,start_date,event_name,purse,winner_share,event_id,course_name,course_num,rank,event_date,avg_skill,x_score,Event_Tier
0,2024,2024-01-14 00:00:00,Sony Open in Hawaii,8300000,1494000,6,Waialae Country Club,6,25,2024-01-14 00:00:00,–0.07,0.92,REGULAR


,dg_id,is_shortlist,decision_context,course_type,decision_score,pct_ytd_avg_sg_total,pct_ytd_made_cut_pct,pct_event_hist_sg,pct_course_hist_sg,final_rank_score,oad_score,pct_sg_total_L12,pct_ev_current_adj,pct_oad_score,decimal_odds,ev_current_adj,ev_future_max,ev_current_to_future_max_ratio,sg_total_L40,sg_total_L24,sg_total_L12,starts_event,made_cut_pct_event,top25_event,top10_event,top5_event,wins_event,prev_finish_num_event,ytd_starts,ytd_made_cut_pct,ytd_top25,ytd_top10,ytd_top5,ytd_wins,ytd_avg_sg_total
0,12423,False,regular|mixed,mixed,0.94,1.00,0.50,0.75,0.86,0.92,0.74,0.90,0.97,0.97,23.00,"375,604.95","225,225.23",1.60,0.62,0.53,1.31,7.00,0.57,3.00,3.00,2.00,0.00,3.00,1.00,1.00,1.00,1.00,1.00,1.00,2.61
1,21554,False,regular|mixed,mixed,0.88,0.91,0.50,0.51,0.50,0.89,0.67,0.86,0.93,0.95,31.00,"278,674.64","378,787.88",0.71,1.53,1.13,1.09,6.00,0.67,2.00,0.00,0.00,0.00,21.00,1.00,1.00,1.00,1.00,1.00,0.00,1.61
2,23014,False,regular|mixed,mixed,0.79,0.97,0.50,0.42,0.36,0.80,0.62,0.73,0.91,0.86,36.00,"239,969.83","490,196.08",0.47,1.02,1.46,0.81,1.00,1.00,0.00,0.00,0.00,0.00,48.00,1.00,1.00,1.00,1.00,1.00,0.00,2.36
3,14459,False,regular|mixed,mixed,0.78,0.95,0.50,0.96,0.94,0.72,0.65,0.57,0.95,0.89,29.00,"297,893.58","378,787.88",0.76,0.97,1.22,0.38,1.00,1.00,1.00,0.00,0.00,0.00,12.00,1.00,1.00,1.00,1.00,1.00,0.00,1.86
4,13997,False,regular|mixed,mixed,0.71,0.49,0.50,0.83,0.77,0.78,0.57,0.73,0.89,0.73,41.00,"210,705.22","190,839.69",1.06,0.85,0.80,0.82,3.00,1.00,1.00,1.00,0.00,0.00,41.00,1.00,1.00,0.00,0.00,0.00,0.00,-0.14
5,14577,False,regular|mixed,mixed,0.70,0.73,0.50,0.42,0.40,0.85,0.57,0.87,0.89,0.68,41.00,"210,705.22","328,947.37",0.62,0.83,0.74,1.16,6.00,0.83,1.00,0.00,0.00,0.00,73.00,1.00,1.00,1.00,0.00,0.00,0.00,0.86
6,14578,False,regular|mixed,mixed,0.69,0.08,0.50,0.77,0.78,0.72,0.65,0.57,0.98,0.92,21.00,"411,376.85","609,756.10",0.65,1.22,0.87,0.31,7.00,0.71,3.00,1.00,1.00,0.00,32.00,1.00,1.00,0.00,0.00,0.00,0.00,-1.89
7,14704,False,regular|mixed,mixed,0.69,0.41,0.50,0.65,0.65,0.71,0.59,0.62,0.87,0.78,46.00,"187,802.48","138,121.55",1.31,0.45,0.61,0.55,6.00,0.83,2.00,2.00,2.00,0.00,4.00,1.00,1.00,0.00,0.00,0.00,0.00,-0.64
8,21756,False,regular|mixed,mixed,0.69,0.82,0.50,0.38,0.35,0.91,0.57,0.96,0.93,0.59,31.00,"278,674.64","609,756.10",0.44,1.51,1.77,1.64,1.00,1.00,0.00,0.00,0.00,0.00,61.00,1.00,1.00,1.00,0.00,0.00,0.00,0.86
9,17576,False,regular|mixed,mixed,0.67,0.49,0.50,0.98,0.97,0.64,0.61,0.45,0.95,0.84,29.00,"297,893.58","378,787.88",0.76,0.68,0.16,-0.08,5.00,1.00,4.00,1.00,1.00,0.00,12.00,1.00,1.00,0.00,0.00,0.00,0.00,-0.14


In [13]:
condensed.info

<bound method DataFrame.info of     dg_id  is_shortlist decision_context course_type  decision_score  pct_ytd_avg_sg_total  pct_ytd_made_cut_pct  pct_event_hist_sg  pct_course_hist_sg  final_rank_score  oad_score  pct_sg_total_L12  pct_ev_current_adj  pct_oad_score  decimal_odds  ev_current_adj ev_future_max  ev_current_to_future_max_ratio  sg_total_L40  sg_total_L24  sg_total_L12  starts_event  made_cut_pct_event  top25_event  top10_event  top5_event  wins_event  prev_finish_num_event  ytd_starts  ytd_made_cut_pct  ytd_top25  ytd_top10  ytd_top5  ytd_wins  ytd_avg_sg_total
0   17780         False  signature|mixed       mixed            0.71                  0.59                  0.07               0.95                 NaN              0.76       0.43              0.90                0.27           0.69         76.00      464,829.52           NaN                             NaN          1.16          1.06          1.73          1.00                1.00         1.00         1.00        

In [18]:
PICK_DG_ID = 8825

picks_log = log_pick(
    SEASON_YEAR,
    EVENT_ID,
    PICK_DG_ID,
    weekly_raw,      # MUST be unfiltered
    odds_df,
    paths.picks_log_path,
)

preview_pick_result(picks_log)

Latest Pick Summary:
---------------------
Player:        Harman, Brian (dg_id 8825)
Finish:        18
Winnings:      $106,102
Cumulative:    $106,102


In [20]:
import pandas as pd
print(pd.read_csv(paths.picks_path).head(10))
print(pd.read_csv(paths.picks_log_path).head(10))


   year  event_id           event_name  dg_id  decimal_odds  ev_current  ev_future_total  ev_current_pct_of_future  finish_num finish_text  winnings  player_name_x  player_name_y    player_name
0  2024         6  Sony Open in Hawaii   8825         19.00  436,842.00     9,358,696.00                       NaN          18         T18    106102            NaN            NaN  Harman, Brian
   year  event_id           event_name  dg_id  decimal_odds  ev_current  ev_future_total  ev_current_pct_of_future  finish_num finish_text  winnings  player_name_x  player_name_y    player_name
0  2024         6  Sony Open in Hawaii   8825         19.00  436,842.00     9,358,696.00                       NaN          18         T18    106102            NaN            NaN  Harman, Brian


In [19]:
print("after log_pick rows:", len(picks_log))
display(picks_log.tail(5))

# also read the file back from disk (proves persistence)
import pandas as pd
re = pd.read_csv(paths.picks_log_path)
print("reloaded rows:", len(re))
display(re.tail(5))

after log_pick rows: 1


,year,event_id,event_name,dg_id,decimal_odds,ev_current,ev_future_total,ev_current_pct_of_future,finish_num,finish_text,winnings,player_name_x,player_name_y,player_name
0,2024,6,Sony Open in Hawaii,8825,19.00,"436,842.00","9,358,696.00",NaN,18,T18,106102,NaN,NaN,"Harman, Brian"


reloaded rows: 1


,year,event_id,event_name,dg_id,decimal_odds,ev_current,ev_future_total,ev_current_pct_of_future,finish_num,finish_text,winnings,player_name_x,player_name_y,player_name
0,2024,6,Sony Open in Hawaii,8825,19.00,"436,842.00","9,358,696.00",NaN,18,T18,106102,NaN,NaN,"Harman, Brian"


In [15]:
FOCUS_DG_IDS = []  # [] => default top 25

focus_ids = FOCUS_DG_IDS or condensed.head(25)["dg_id"].astype(int).tolist()
weekly_f = filter_top_tables_by_ids(weekly, focus_ids)

display(weekly_f["table_performance_top50"])
display(weekly_f["table_event_history_top50"])
display(weekly_f["table_ytd_top50"])

,dg_id,oad_score,pct_sg_total_L12,pct_ev_current_adj,pct_oad_score,final_rank_score,decision_score,decision_context,decimal_odds,ev_current,ev_current_adj,ev_future_total,ev_future_max,ev_future_total_adj,ev_future_max_adj,ev_current_to_future_max_ratio,z_sg_recent,z_ytd,z_history,z_ev_current,course_fit_score,is_shortlist,tagged_here,pattern_score_winner,pattern_score_top5,sg_total_L40,sg_app_L40,sg_putt_L40,round_score_L40,sg_total_L24,sg_app_L24,sg_putt_L24,round_score_L24,sg_total_L12,sg_app_L12,sg_putt_L12,round_score_L12
11,17780,0.43,0.90,0.27,0.69,0.76,0.71,signature|mixed,76.00,"263,158.00","464,829.52",NaN,NaN,NaN,NaN,NaN,0.69,0.29,0.44,0.06,NaN,False,False,9,8,1.16,0.36,0.52,69.42,1.06,0.55,0.29,68.58,1.73,0.75,0.71,68.17
16,19483,0.48,0.73,0.51,0.73,0.70,0.70,signature|mixed,51.00,"392,157.00","692,687.13",NaN,NaN,NaN,NaN,NaN,0.64,0.55,0.50,0.11,NaN,False,False,8,10,1.05,0.11,0.53,69.90,1.27,-0.10,1.08,69.62,0.96,0.36,1.14,69.33
21,18841,0.47,0.64,0.93,0.71,0.70,0.62,signature|mixed,17.00,"1,176,471.00","2,078,061.38",NaN,NaN,NaN,NaN,NaN,0.54,0.36,0.58,0.42,NaN,True,False,4,4,0.77,0.60,0.03,70.08,0.56,0.43,0.03,69.62,0.71,0.83,-0.40,69.58
29,9771,0.42,0.71,0.66,0.67,0.70,0.60,signature|mixed,41.00,"487,805.00","861,635.21",NaN,NaN,NaN,NaN,NaN,0.54,0.51,0.41,0.15,NaN,False,False,7,7,0.60,-0.28,0.69,69.95,0.57,-0.17,0.62,69.50,0.85,-0.39,0.90,69.33
0,13872,0.42,0.67,0.46,0.63,0.63,0.56,signature|mixed,56.00,"357,143.00","630,840.06",NaN,NaN,NaN,NaN,NaN,0.56,0.36,0.55,0.10,NaN,False,False,5,5,0.86,0.27,0.22,69.92,0.65,0.30,0.04,69.50,0.77,0.35,-0.15,68.83
30,18103,0.35,0.43,0.33,0.43,0.41,0.47,signature|mixed,71.00,"281,690.00","497,563.99",NaN,NaN,NaN,NaN,NaN,0.48,0.39,0.36,0.07,NaN,False,False,9,6,0.80,0.11,0.42,69.97,0.42,-0.03,0.32,69.50,0.19,0.28,-0.02,69.42


,dg_id,starts_event,made_cuts_event,made_cut_pct_event,top25_event,top10_event,top5_event,wins_event,prev_finish_num_event,avg_score_event,avg_sg_total_event
11,17780,1.00,1.00,1.00,1.00,1.00,0.00,0.00,8.00,68.75,1.50
16,19483,3.00,3.00,1.00,3.00,1.00,0.00,0.00,15.00,68.42,1.19
21,18841,4.00,4.00,1.00,2.00,1.00,1.00,1.00,1.00,69.12,1.03
29,9771,5.00,5.00,1.00,2.00,1.00,1.00,0.00,45.00,70.10,-0.48
0,13872,6.00,6.00,1.00,1.00,1.00,1.00,1.00,29.00,69.46,-0.17
30,18103,1.00,1.00,1.00,1.00,0.00,0.00,0.00,12.00,69.00,1.25


,dg_id,ytd_starts,ytd_made_cuts,ytd_made_cut_pct,ytd_top25,ytd_top10,ytd_top5,ytd_wins,ytd_avg_score,ytd_avg_sg_total
11,17780,21,14,0.67,11,6,3,1,69.52,0.87
16,19483,19,16,0.84,10,7,2,0,69.65,0.99
21,18841,15,12,0.80,7,2,2,0,70.09,0.54
29,9771,20,17,0.85,9,5,1,0,69.71,0.74
0,13872,20,16,0.80,7,2,2,0,70.35,0.38
30,18103,23,18,0.78,14,4,3,0,69.46,1.12


In [16]:
print_event_history_for_player(weekly, 16283)

dg_focus = [16283]
slices = subset_weekly_for_players(weekly, dg_focus)

display(plot_ev_current_vs_future_max(slices["performance"]))

plot_course_vs_players_radar_from_skills(
    weekly,
    course_fit_df,
    player_skills_df,
    dg_focus,
)

Event history for dg_id 16283 at BMW Championship (event_id 28), years 2017–2023


,year,finish_num,fin_text,sg_event
0,2023,37.00,T37,-4.54



Summary:
Starts: 1
Best finish: 37.0
Top-10s: 0
Top-25s: 0
Wins: 0
No data in performance slice.


,dg_id,oad_score,pct_sg_total_L12,pct_ev_current_adj,pct_oad_score,final_rank_score,decision_score,decision_context,decimal_odds,ev_current,ev_current_adj,ev_future_total,ev_future_max,ev_future_total_adj,ev_future_max_adj,ev_current_to_future_max_ratio,z_sg_recent,z_ytd,z_history,z_ev_current,course_fit_score,is_shortlist,tagged_here,pattern_score_winner,pattern_score_top5,sg_total_L40,sg_app_L40,sg_putt_L40,round_score_L40,sg_total_L24,sg_app_L24,sg_putt_L24,round_score_L24,sg_total_L12,sg_app_L12,sg_putt_L12,round_score_L12


No course_fit row found for course_num=406


In [17]:
picks_with_names = add_player_names_to_picks_file(paths.picks_path, rounds_df, overwrite=True)
display(picks_with_names.head(35))

MergeError: Passing 'suffixes' which cause duplicate columns {'player_name_x'} is not allowed.

In [10]:
row = weekly_raw["schedule_row"].iloc[0]
eid = int(row["event_id"])
ename = row["event_name"]
sched_date = pd.to_datetime(row["event_date"], errors="coerce")

print("event:", ename, "event_id:", eid)
print("schedule event_date:", sched_date)

# what does odds/results say for completion date?
tmp = odds_df[(odds_df["year"] == SEASON_YEAR) & (odds_df["event_id"] == eid)].copy()
if "event_completed" in tmp.columns:
    tmp["event_completed"] = pd.to_datetime(tmp["event_completed"], errors="coerce")
    print("odds event_completed (min/max):", tmp["event_completed"].min(), tmp["event_completed"].max())
else:
    print("odds_df has no event_completed column")

# what as_of_date are we passing into YTD / features?
# (you have to print whatever variable you pass; if build_weekly_view constructs it internally, add it to weekly dict once)
print("weekly keys:", weekly_raw.keys())


event: The Open Championship event_id: 100
schedule event_date: 2025-07-20 00:00:00
odds event_completed (min/max): 2025-07-20 00:00:00 2025-07-20 00:00:00
weekly keys: dict_keys(['schedule_row', 'field', 'rolling', 'event_history', 'ytd', 'course_fit', 'course_history', 'ev_current', 'ev_future', 'summary', 'table_performance', 'table_event_history', 'table_ytd', 'table_performance_top50', 'table_event_history_top50', 'table_ytd_top50', 'pattern_candidates', 'patterns_summary_text', 'patterns_winners_features', 'patterns_top5_features'])


In [11]:
print("odds_df rows for player before cutoff:",
      odds_df[(odds_df["dg_id"] == 8825) & (pd.to_datetime(odds_df["event_completed"]) < cutoff_date)].shape[0]
)

NameError: name 'cutoff_date' is not defined

In [7]:
doc = "/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2025.csv"
df = pd.read_csv(doc)

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311040 entries, 0 to 311039
Data columns (total 37 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   tour              311040 non-null  object 
 1   year              311040 non-null  int64  
 2   season            311040 non-null  int64  
 3   event_completed   311040 non-null  object 
 4   event_name        311040 non-null  object 
 5   event_id          311040 non-null  int64  
 6   player_name       311040 non-null  object 
 7   dg_id             311040 non-null  int64  
 8   fin_text          311040 non-null  object 
 9   round_num         311040 non-null  int64  
 10  course_name       311040 non-null  object 
 11  course_num        311040 non-null  int64  
 12  course_par        306528 non-null  float64
 13  start_hole        298526 non-null  float64
 14  teetime           298526 non-null  object 
 15  round_score       310116 non-null  float64
 16  sg_putt           13

In [3]:
import pandas as pd
oad = "/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2025.csv"
oad_df = pd.read_csv(oad)
oad_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311040 entries, 0 to 311039
Data columns (total 37 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   tour              311040 non-null  object 
 1   year              311040 non-null  int64  
 2   season            311040 non-null  int64  
 3   event_completed   311040 non-null  object 
 4   event_name        311040 non-null  object 
 5   event_id          311040 non-null  int64  
 6   player_name       311040 non-null  object 
 7   dg_id             311040 non-null  int64  
 8   fin_text          311040 non-null  object 
 9   round_num         311040 non-null  int64  
 10  course_name       311040 non-null  object 
 11  course_num        311040 non-null  int64  
 12  course_par        306528 non-null  float64
 13  start_hole        298526 non-null  float64
 14  teetime           298526 non-null  object 
 15  round_score       310116 non-null  float64
 16  sg_putt           13

/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_85995/1272692513.py:3: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  oad_df = pd.read_csv(oad)


In [2]:
weekly_raw = build_weekly_view(SEASON_YEAR, EVENT_ID)
weekly_raw["summary"][["dg_id","decision_score","decision_context","pct_form","pct_ytd_avg_sg_total","pct_event_hist_sg","pct_course_hist_sg"]].head(10)


NameError: name 'EVENT_ID' is not defined

In [18]:
weekly_raw = build_weekly_view(SEASON_YEAR, EVENT_ID)

print("schedule_row type:", type(weekly_raw.get("schedule_row")))
print("schedule_row columns:", list(getattr(weekly_raw.get("schedule_row"), "columns", []))[:50])

print("summary cols contains decision_context:", "decision_context" in weekly_raw["summary"].columns)
print("summary cols contains course_type:", "course_type" in weekly_raw["summary"].columns)

print("summary decision_context value counts:")
print(weekly_raw["summary"]["decision_context"].value_counts(dropna=False).head(10))

print("summary course_type value counts:")
print(weekly_raw["summary"]["course_type"].value_counts(dropna=False).head(10))

schedule_row type: <class 'pandas.core.frame.DataFrame'>
schedule_row columns: ['year', 'start_date', 'event_name', 'purse', 'winner_share', 'event_id', 'course_name', 'course_num', 'rank', 'event_date', 'avg_skill', 'x_score', 'Event_Tier']
summary cols contains decision_context: True
summary cols contains course_type: True
summary decision_context value counts:
decision_context
signature|mixed    50
Name: count, dtype: int64
summary course_type value counts:
course_type
mixed    50
Name: count, dtype: int64


In [1]:
import pandas as pd
from pathlib import Path

SCHEDULE_XLSX = {
    2024: Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/OAD_2024.xlsx"),
    2025: Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/OAD_2025.xlsx"),
}

season = 2025
df = pd.read_excel(SCHEDULE_XLSX[season])
print(df.columns.tolist())


['year', 'start_date', 'event_name', 'purse', 'winner_share', 'event_id', 'course_name', 'course_num', 'rank']


In [5]:
import pandas as pd

league2025 = "/Users/joshmacbook/python_projects/OAD/Data/Clean/Leagues/2025_small_normalized.csv"
league2025 = pd.read_csv(league2025)

league2025.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1483 entries, 0 to 1482
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   season        1483 non-null   int64  
 1   league_id     1483 non-null   object 
 2   event_name    1483 non-null   object 
 3   event_date    1483 non-null   object 
 4   event_key     1483 non-null   object 
 5   event_id      1483 non-null   int64  
 6   entry_id      1483 non-null   object 
 7   username      1483 non-null   object 
 8   player_name   1483 non-null   object 
 9   dg_id         1483 non-null   float64
 10  raw_winnings  1483 non-null   float64
dtypes: float64(2), int64(2), object(7)
memory usage: 127.6+ KB


In [21]:
from Scripts.weekly_view import build_weekly_view

weekly_raw = build_weekly_view(2024, 6)  # same event as your pick row
weekly_filtered, used_ids = apply_used_filter_to_weekly(weekly_raw.copy(), paths.picks_path)

print("used_ids:", used_ids)
print("used_ids contains 8825:", 8825 in set(map(int, used_ids)))

raw_ids = set(weekly_raw["summary"]["dg_id"].dropna().astype(int))
flt_ids = set(weekly_filtered["summary"]["dg_id"].dropna().astype(int))

print("raw has 8825:", 8825 in raw_ids)
print("filtered has 8825:", 8825 in flt_ids)


used_ids: {8825}
used_ids contains 8825: True
raw has 8825: True
filtered has 8825: True


In [23]:
def find_dgid(weekly_obj, dgid=8825):
    out = {}
    for k, v in weekly_obj.items():
        if isinstance(v, pd.DataFrame) and "dg_id" in v.columns:
            s = pd.to_numeric(v["dg_id"], errors="coerce").fillna(-1).astype(int)
            out[k] = int((s == int(dgid)).sum())
    return out

print("used_ids:", used_ids)
print("counts by table:", find_dgid(weekly_raw, 8825))
print("counts by table (filtered):", find_dgid(weekly, 8825))

# also check summary rank position if still present
s = weekly["summary"].copy()
s["dg_id"] = pd.to_numeric(s["dg_id"], errors="coerce").fillna(-1).astype(int)
hit = s[s["dg_id"] == 8825]
print("in filtered summary rows:", len(hit))
if not hit.empty:
    display(hit.head(5))


used_ids: {8825}
counts by table: {'field': 1, 'rolling': 1, 'event_history': 1, 'ytd': 1, 'course_fit': 1, 'course_history': 1, 'ev_current': 1, 'ev_future': 31, 'summary': 1, 'table_performance': 1, 'table_event_history': 1, 'table_ytd': 1, 'table_performance_top50': 1, 'table_event_history_top50': 1, 'table_ytd_top50': 1, 'pattern_candidates': 1, 'patterns_winners_features': 0, 'patterns_top5_features': 0}
counts by table (filtered): {'field': 1, 'rolling': 1, 'event_history': 1, 'ytd': 1, 'course_fit': 1, 'course_history': 1, 'ev_current': 1, 'ev_future': 31, 'summary': 1, 'table_performance': 0, 'table_event_history': 0, 'table_ytd': 0, 'table_performance_top50': 0, 'table_event_history_top50': 0, 'table_ytd_top50': 0, 'pattern_candidates': 0, 'patterns_winners_features': 0, 'patterns_top5_features': 0}
in filtered summary rows: 1


,year,event_id,dg_id,player_name,Event_Tier,close_odds,sg_total_L40,sg_app_L40,sg_arg_L40,sg_putt_L40,driving_dist_L40,driving_acc_L40,round_score_L40,sg_total_L24,sg_app_L24,sg_arg_L24,sg_putt_L24,driving_dist_L24,driving_acc_L24,round_score_L24,sg_total_L12,sg_app_L12,sg_arg_L12,sg_putt_L12,driving_dist_L12,driving_acc_L12,round_score_L12,starts_event,made_cuts_event,made_cut_pct_event,top25_event,top10_event,top5_event,wins_event,prev_finish_num_event,prev_finish_text_event,avg_score_event,avg_sg_total_event,ytd_made_cuts,ytd_top25,ytd_top10,ytd_top5,ytd_wins,ytd_starts,ytd_made_cut_pct,ytd_avg_score,ytd_avg_sg_total,course_fit_score,decimal_odds,ev_current,ev_future_total,ev_future_max,course_starts,course_avg_sg_total,course_avg_score,course_num,course_type,ev_current_vs_future_max_pct,ev_current_pct_of_future,is_liv,tag_event_1,tag_event_2,tag_event_3,tag_event_4,is_shortlist,tagged_here,ev_current_adj,ev_future_total_adj,ev_future_max_adj,ev_current_to_future_max_ratio,z_sg_recent,z_ytd,event_hist_raw,event_hist_z,history_metric,z_history,z_ev_current,oad_score,pct_sg_total_L12,pct_ev_current_adj,pct_oad_score,pct_form,final_rank_score,pct_ytd_avg_sg_total,pct_ytd_made_cut_pct,pct_event_hist_sg,pct_course_hist_sg,decision_score,decision_context,pattern_score_winner,pattern_flag_winner,pattern_score_top5,pattern_flag_top5
12,2024,6,8825,"Harman, Brian",REGULAR,19.00,1.43,0.15,0.33,1.03,291.57,0.63,67.92,0.72,-0.02,0.44,0.59,297.72,0.57,68.17,0.84,0.01,0.53,0.48,303.77,0.60,67.92,7.00,6.00,0.86,2.00,1.00,1.00,0.00,32.00,32,68.36,0.33,1.00,1.00,1.00,1.00,0.00,1.00,1.00,66.75,1.61,-0.11,19.00,"436,842.11","9,358,695.65","543,478.26",7.00,0.60,68.04,6,mixed,0.80,0.05,NaN,<NA>,<NA>,<NA>,<NA>,False,False,"454,679.68","9,740,839.27","565,670.11",0.80,0.94,0.40,0.67,0.81,0.81,0.81,0.89,0.75,0.75,0.99,1.00,0.75,0.85,0.91,0.50,0.57,0.58,0.89,regular|mixed,7,True,7,True
